# 2026 제출 모델 번들 생성

이 노트북은 **제출 전에 한 번만** 실행하는 학습용 노트북입니다. 2019~2025 학습자료로 동결된 3-seed 모델을 재학습하고, Kaggle Dataset으로 업로드할 ZIP을 Google Drive에 만듭니다. 제출 노트북에는 이 학습 코드가 들어가지 않습니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/SME_DATA_submission_2026"
BRANCH = "agent/submission-2026-end-to-end"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/tswaincae1221/SME_DATA.git", REPO_DIR,
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "catboost==1.2.8", "torch", "scikit-learn", "pandas", "numpy",
], check=True)
print("repo:", REPO_DIR)


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SME_DATA")
MASTER_CSV = DRIVE_ROOT / "processed_station_features/final_train_dataset_19to25_master.csv"
SHORTTERM_CSV = DRIVE_ROOT / "processed_station_features/shortterm_12to14_data/incremental_12to14_tables/shortterm_long_2019to2025.csv"

station_candidates = sorted(DRIVE_ROOT.rglob("station_list.csv"))
if not station_candidates:
    raise FileNotFoundError("SME_DATA 아래 station_list.csv를 찾지 못했습니다")
STATION_CSV = station_candidates[0]
OUTPUT_DIR = DRIVE_ROOT / "submission_2026/model_bundle_v1"

for path in [MASTER_CSV, SHORTTERM_CSV, STATION_CSV]:
    if not path.is_file():
        raise FileNotFoundError(path)
print("master:", MASTER_CSV)
print("shortterm:", SHORTTERM_CSV)
print("station:", STATION_CSV)
print("output:", OUTPUT_DIR)


In [ ]:
import subprocess, sys

oof_root = Path(REPO_DIR) / "assets/submission_2026"
cmd = [
    sys.executable, "-u", str(Path(REPO_DIR) / "scripts/build_submission_2026_bundle.py"),
    "--master-csv", str(MASTER_CSV),
    "--shortterm-long-csv", str(SHORTTERM_CSV),
    "--station-list", str(STATION_CSV),
    "--baseline-oof-dirs",
    str(oof_root / "seed_42"), str(oof_root / "seed_43"), str(oof_root / "seed_44"),
    "--output-dir", str(OUTPUT_DIR),
    "--threads", "4", "--device", "auto",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import json, shutil

manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))
required = [OUTPUT_DIR / "manifest.json", OUTPUT_DIR / "submission_2026_inference.py"]
for seed in manifest["seeds"]:
    root = OUTPUT_DIR / f"seed_{seed}"
    required += [
        root / "base_catboost_TA.cbm", root / "base_catboost_HM.cbm",
        root / "residual_lstm_TA.pt", root / "residual_lstm_HM.pt",
        root / "ta_residual_ridge.joblib", root / "june_direct_ridge_TA.joblib",
        root / "team_spatial_catboost_HM.cbm",
    ]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError("bundle files missing:
" + "
".join(missing))

archive_base = str(OUTPUT_DIR.parent / "sme_submission_2026_model_bundle_v1")
archive = shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Kaggle Dataset upload ZIP:", archive)
print(json.dumps(manifest["historical_report"], ensure_ascii=False, indent=2))
